In [66]:
import os
from dotenv import load_dotenv

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client import models

In [67]:
load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qdrant_client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "renovation_docs"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

In [68]:
OUT_OF_DOMAIN_MESSAGE = (
    "Я помогаю только с вопросами о ремонте и услугах компании."
)

CONTACTS_RESPONSE = (
    "Связаться с менеджером можно по телефону +7(499)372-47-47. "
    "Так же можно оставить заявку в приложении или на сайте."
)

MIN_RETRIEVAL_SCORE = 0.35

## Embedding query


In [69]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=query,
    )
    return response.data[0].embedding

## Qdrant Retreival

In [70]:
def retrieve_candidates(query: str, top_k: int = 8):
    query_vector = embed_query(query)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return response.points

In [71]:
def retrieve_candidates_once(query: str, top_k: int = 8, source_mode: str = "all"):
    query_vector = embed_query(query)

    query_filter = None

    if source_mode == "pricing_only":
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="document_type",
                    match=models.MatchValue(value="pricing"),
                )
            ]
        )
    elif source_mode == "website_only":
        query_filter = models.Filter(
            must_not=[
                models.FieldCondition(
                    key="document_type",
                    match=models.MatchValue(value="pricing"),
                )
            ]
        )

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=query_filter,
    )

    return response.points

In [72]:
def retrieve_candidates(query: str, top_k: int = 8):
    source_mode = "website_only" if is_broad_repair_question(query) else "all"

    candidates = retrieve_candidates_once(
        query=query,
        top_k=top_k,
        source_mode=source_mode,
    )

    if should_retry_with_rewrite(query, candidates):
        rewritten_query = rewrite_query_with_llm(query)

        candidates = retrieve_candidates_once(
            query=rewritten_query,
            top_k=top_k,
            source_mode=source_mode,
        )

    return candidates

## Context

In [73]:
def build_context(points) -> str:
    parts = []

    for point in points:
        payload = point.payload or {}
        text = (payload.get("text") or "").strip()
        if text:
            parts.append(text)

    return "\n\n---\n\n".join(parts)

In [74]:
def rerank_chunks(question: str, candidates, top_n: int = 3):
    scored = []

    for point in candidates:
        payload = point.payload or {}
        chunk_text = (payload.get("text") or "").strip()

        if not chunk_text:
            continue

        rerank_prompt = f"""
Вопрос:
{question}

Фрагмент:
{chunk_text}

Оцени, насколько этот фрагмент помогает ответить на вопрос пользователя.

Верни только одно число от 0 до 10:
0 — фрагмент не помогает
10 — фрагмент максимально полезен
""".strip()

        response = openai_client.responses.create(
            model=CHAT_MODEL,
            input=[
                {
                    "role": "system",
                    "content": "Ты оцениваешь релевантность фрагмента для ответа на вопрос. Верни только число от 0 до 10."
                },
                {
                    "role": "user",
                    "content": rerank_prompt,
                },
            ],
            max_output_tokens=16,
            temperature=0,
        )

        raw_score = (response.output_text or "").strip()

        try:
            score = float(raw_score.replace(",", "."))
        except ValueError:
            score = 0.0

        scored.append((score, point))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [point for score, point in scored[:top_n]]

In [75]:
def is_broad_repair_question(question: str) -> bool:
    q = question.lower()

    broad_hints = [
        "квартира", "однушка", "двушка", "трешка", "студия",
        "ремонт квартиры", "стоимость ремонта", "сколько выйдет",
        "сколько будет стоить", "примерная стоимость",
        "сколько примерно", "капитальный ремонт", "дизайнерский ремонт",
        "вторичка", "новостройка", "без выезда", "примерно понять цену",
        "примерно понять стоимость", "порядок цен", "сколько примерно",
        "ориентировочная стоимость",
        "примерная стоимость", "цена ремонта", "стоимость ремонта",
    ]

    return any(hint in q for hint in broad_hints)

In [99]:
def looks_like_repair_question(question: str) -> bool:
    q = question.lower()

    repair_hints = [
        "ремонт", "квартира", "однушка", "двушка", "трешка", "студия",
        "кухн", "ванн", "сануз", "новострой", "вторич",
        "смет", "стоим", "цена", "цен", "порядок цен",
        "материал", "дизайн", "демонтаж", "монтаж", "отделк",
        "замер", "замерщик", "договор", "гарант", "рассроч",
        "менеджер", "контакт", "связат", "написать", "заявк",
        "телефон", "номер", "ватсап", "whatsapp", "телега", "telegram",
        "под ключ", "чернов", "чистов", "электрик", "сантех",
        "офис", "коммерц", "нежил", "коттедж", "дом",
        "москва", "область", "регион", "без выезда",
        "работаете", "работаете ли", "где работаете", "по москве", "по области",
    ]

    return any(hint in q for hint in repair_hints)

In [100]:
def is_contact_question(question: str) -> bool:
    q = question.lower()

    contact_hints = [
        "контакт", "контакты", "связаться", "связь",
        "менеджер", "телефон", "номер", "позвонить",
        "написать", "заявка", "заявку", "оставить заявку",
        "ватсап", "whatsapp", "телега", "telegram",
    ]

    return any(hint in q for hint in contact_hints)

In [101]:
def answer_contact_question(question: str):
    return CONTACTS_RESPONSE, [], ""

In [102]:
def is_pricing_point(point) -> bool:
    payload = point.payload or {}
    return payload.get("document_type") == "pricing"


def should_retry_with_rewrite(question: str, candidates) -> bool:

    if not looks_like_repair_question(question):
        return False

    if not candidates:
        return True

    top_score = candidates[0].score or 0.0

    if top_score < 0.35:
        return True

    if is_broad_repair_question(question):
        top3 = candidates[:3]
        pricing_count = sum(1 for point in top3 if is_pricing_point(point))

        if pricing_count >= 2:
            return True

        if top_score < 0.52:
            return True

    return False

## Prompt

In [103]:
SYSTEM_PROMPT = """
Ты — помощник компании по ремонту квартир.

Правила:
1. Отвечай только по переданному контексту.
2. Если в контексте есть точная цена, срок, гарантия, формат работы или контакт — используй их в ответе.
3. Если вопрос про стоимость, считай только из явных данных в контексте. Не придумывай свои цифры, площади, проценты, сроки или диапазоны.
4. Если в контексте есть цена за м² и пользователь указал площадь, можешь посчитать примерную стоимость.
5. Если в контексте есть цена за м², но площадь не указана, можно сказать только, что стоимость считается от цены за м², без выдуманных площадей.
6. Если услуга, тип объекта, регион, рассрочка, замер, закупка материалов или другой факт не подтвержден в контексте явно, не отвечай уверенно "да" или "нет". Вместо этого скажи, что это лучше уточнить у менеджера.
7. Если информации недостаточно, скажи по-человечески, что лучше уточнить детали у менеджера.
8. Если вопрос вообще не связан с ремонтом и услугами компании, коротко скажи, что ты помогаешь только по вопросам ремонта и услуг компании.
9. Не используй общие знания, если их нет в контексте.
10. Не упоминай базу знаний.
11. Отвечай по-русски, коротко и естественно, максимум 4-5 предложений.
12. Если вопрос про общую стоимость ремонта, порядок цен или оценку без выезда, не используй отдельные цены на конкретные работы как ответ на общий вопрос.
13. Не называй компанию по имени, если пользователь сам об этом не спрашивал.
14. Не перефразируй факты из контекста так, чтобы менялся смысл. Например, "фиксированная смета" не означает "фиксированная цена".
""".strip()

## Rewrite query

In [104]:
def rewrite_query_with_llm(question: str) -> str:
    prompt = f"""
Перепиши вопрос пользователя в нормальный поисковый запрос для базы знаний по ремонту квартир.

Правила:
1. Сохрани исходный смысл.
2. Исправь разговорные слова и опечатки.
3. Замени бытовые формулировки на более стандартные.
4. Не добавляй новых фактов.
5. Верни только одну короткую переписанную фразу, без пояснений.

Вопрос:
{question}
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {
                "role": "system",
                "content": "Ты переписываешь пользовательский запрос для поиска по базе знаний. Верни только переписанный запрос.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_output_tokens=40,
        temperature=0,
    )

    rewritten = (response.output_text or "").strip()
    return rewritten or question

## Generator

In [105]:
def answer_with_rag(question: str, top_k: int = 8, top_n: int = 3):

    if is_contact_question(question):
        return answer_contact_question(question)

    if not looks_like_repair_question(question):
        return OUT_OF_DOMAIN_MESSAGE, [], ""

    candidates = retrieve_candidates(question, top_k=top_k)

    if not candidates:
        return OUT_OF_DOMAIN_MESSAGE, [], ""

    best_score = candidates[0].score or 0.0
    if best_score < MIN_RETRIEVAL_SCORE:
        return OUT_OF_DOMAIN_MESSAGE, candidates, ""

    best_points = rerank_chunks(question, candidates, top_n=top_n)

    if not best_points:
        return "Не могу сейчас ответить точно. Лучше уточнить детали у менеджера.", [], ""

    context = build_context(best_points)

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}

Ответь по правилам.
Если вопрос про примерную стоимость и в контексте есть ориентир по цене, дай примерный расчет.
Если площадь не указана, можно привести короткий пример для типовой площади.
Если данных мало, скажи, что лучше уточнить детали у менеджера.
Если точного ответа нет, не выдумывай.
Не упоминай название компании без необходимости.
Не заменяй формулировки из контекста на более сильные утверждения.
Если в контексте сказано "фиксированная смета", не называй это "фиксированной ценой".
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, best_points, context

In [106]:
def answer_with_baseline(question: str, top_k: int = 5):
    points = retrieve_candidates(question, top_k=top_k)
    context = build_context(points[:top_k])

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}

Ответь по правилам.
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, points[:top_k], context

In [107]:
def debug_rewrite_flow(question: str, top_k: int = 8):
    source_mode = "website_only" if is_broad_repair_question(question) else "all"
    first = retrieve_candidates_once(question, top_k=top_k, source_mode=source_mode)

    print("QUESTION:", question)
    print("SOURCE MODE:", source_mode)
    print("FIRST TOP SCORE:", first[0].score if first else None)
    print("REWRITE NEEDED:", should_retry_with_rewrite(question, first))

    if should_retry_with_rewrite(question, first):
        rewritten = rewrite_query_with_llm(question)
        print("REWRITTEN:", rewritten)

        second = retrieve_candidates_once(rewritten, top_k=top_k, source_mode=source_mode)
        print("SECOND TOP SCORE:", second[0].score if second else None)
        return first, second

    return first, first

# Tests

In [108]:
def print_points(points, max_chars: int = 500):
    for i, point in enumerate(points, start=1):
        payload = point.payload or {}
        print("=" * 100)
        print(f"RESULT #{i}")
        print(f"score: {getattr(point, 'score', None)}")
        print(f"title: {payload.get('title')}")
        print(f"url: {payload.get('url')}")
        print()
        print((payload.get("text") or "")[:max_chars])
        print()

In [109]:
def compare_answers(question: str, baseline_top_k: int = 5, rerank_top_k: int = 8, rerank_top_n: int = 3):
    print("#" * 120)
    print("QUESTION:", question)
    print()

    baseline_answer, baseline_points, _ = answer_with_baseline(question, top_k=baseline_top_k)
    print("BASELINE ANSWER:")
    print(baseline_answer)
    print()

    rerank_answer, rerank_points, _ = answer_with_rag(question, top_k=rerank_top_k, top_n=rerank_top_n)
    print("RERANK ANSWER:")
    print(rerank_answer)
    print()

    print("BASELINE POINTS:")
    print_points(baseline_points, max_chars=350)

    print("RERANK POINTS:")
    print_points(rerank_points, max_chars=350)

In [20]:
test_questions = [
    "Сколько примерно стоит ремонт квартиры 40 квадратов?",
    "Хочу понять, во сколько выйдет ремонт однушки",
    "Сколько стоит сделать ремонт на кухне?",
    "А ванную под ключ делаете?",
    "Есть ли у вас дизайнерский ремонт?",
    "Вы делаете ремонт в новостройке?",
    "Сколько по времени обычно идет ремонт квартиры?",
    "Можно примерно понять цену без выезда?",
    "От чего вообще зависит стоимость ремонта?",
    "Есть ли у вас гарантия на работы?",
    "Вы по договору работаете?",
    "Можно у вас заказать ремонт + дизайн сразу?",
    "Делаете ли вы капитальный ремонт?",
    "Скажите пожалуйста, вы только квартиры делаете?",
    "А если вторичка, вы с такими квартирами работаете?",
    "Сколько будет стоить ремонт двушки 54 метра?",
    "Можно примерную цену на ремонт кухни 12 метров?",
    "А если квартира убитая совсем, цена сильно выше будет?",
    "Сколько выйдет черновой ремонт?",
    "Можно ли по фото примерно посчитать стоимость?",
    "Есть ли у вас рассрочка?",
    "Работаете ли вы поэтапно или только под ключ?",
    "Можно ли сделать сначала санузел, а потом остальное?",
    "Вы закупкой материалов занимаетесь?",
    "Кто у вас контролирует ремонт на объекте?",
    "Дай контакты менеджера",
    "Как с вами связаться?",
    "Можно номер телефона",
    "Куда написать чтобы обсудить ремонт?",
    "А ватсап есть?",
    "С кем можно поговорить по расчету стоимости?",
    "Хочу связаться с менеджером",
    "Где оставить заявку?",
    "Как вам написать?",
    "Есть у вас телега или вотсап?",
    "Вы делаете ремонт складских помещений?",
    "А офисы ремонтируете?",
    "С коммерцией работаете?",
    "Можно у вас заказать ремонт дома, не квартиры?",
    "Вы делаете только квартиры или коттеджи тоже?",
    "А нежилые помещения берете в работу?",
    "Можно у вас заказать просто дизайн без ремонта?",
    "Только Москва или область тоже?",
    "Вы сами материалы покупаете или это на клиенте?",
    "Можно вызвать замерщика?",
    "Какая погода в Москве?",
    "Объясни квантовую механику простыми словами",
    "Напиши код на Python для телеграм бота",
    "Кто выиграл чемпионат мира?",
    "Как приготовить карбонару?",
    "Посоветуй фильм на вечер",
    "Сколько будет 245 умножить на 17?",
    "Расскажи анекдот",
    "Что лучше, айфон или самсунг?",
    "Помоги выбрать ноутбук",
    "скок стоит ремонт кухни примерно",
    "а вы ванну под ключ делаете или нет?",
    "мне нужен ремнот квартиры 40м2 сколько это выйдет",
    "если у меня вторичка убитая совсем это сильно дороже?",
    "можно без замера хотябы примерно понять цену",
    "дайте номер менеджера пж",
    "а с вами можно в ватсапе списаться?",
    "а офисы вы тоже делаете или тока квартиры?",
    "а если я просто хочу узнать порядок цен",
    "вы вобще по москве только работаете?",
]

In [110]:
check_questions = [
    "Можно примерно понять цену без выезда?",
    "Есть ли у вас рассрочка?",
    "Как с вами связаться?",
    "А ватсап есть?",
    "Как вам написать?",
    "Только Москва или область тоже?",
    "а если я просто хочу узнать порядок цен",
    "вы вобще по москве только работаете?",
    "А офисы ремонтируете?",
    "Можно у вас заказать ремонт дома, не квартиры?",
    "Дай контакты менеджера",
    "Как с вами связаться?",
    "Можно номер телефона",
    "А ватсап есть?",
    "Как вам написать?",
    "Есть у вас телега или вотсап?",
]

In [113]:
for q in test_questions:
    answer, _, _ = answer_with_rag(q)
    print(f"Question: {q} \nAnswer: {answer}\n")

Question: Сколько примерно стоит ремонт квартиры 40 квадратов? 
Answer: Стоимость ремонта квартиры 40 квадратов при цене от 15 000 ₽/м² составляет примерно 600 000 ₽. Точную сумму лучше уточнить у менеджера, так как зависит от выбранных материалов и объема работ.

Question: Хочу понять, во сколько выйдет ремонт однушки 
Answer: Стоимость ремонта однушки в Москве считается от цены за м², но точную сумму лучше уточнить у менеджера после замеров и обсуждения проекта. Для типовой однокомнатной квартиры площадь обычно составляет около 30-40 м², и цена может варьироваться в зависимости от выбранных материалов и работ. Рекомендуется запросить смету, чтобы получить более точную оценку.

Question: Сколько стоит сделать ремонт на кухне? 
Answer: Стоимость ремонта кухни зависит от объема работ и выбранных материалов. В контексте есть цена за демонтаж кухонной мойки — 670 рублей за штуку, а установка — 1350 рублей за штуку. Для общей стоимости ремонта лучше уточнить у менеджера, так как она зависи

## Debug questions

In [26]:
debug_questions = [
    "Хочу понять, во сколько выйдет ремонт однушки",
    "скок стоит ремонт двушки",
    "если вторичка убитая совсем это сильно дороже?",
    "Сколько стоит демонтаж плитки?",
    "Объясни квантовую механику",
]

In [29]:
for q in debug_questions:
    print("=" * 100)
    debug_rewrite_flow(q)
    print()

QUESTION: Хочу понять, во сколько выйдет ремонт однушки
SOURCE MODE: website_only
FIRST TOP SCORE: 0.43340844
REWRITE NEEDED: True
REWRITTEN: Стоимость ремонта однокомнатной квартиры
SECOND TOP SCORE: 0.5639408

QUESTION: скок стоит ремонт двушки
SOURCE MODE: all
FIRST TOP SCORE: 0.46564758
REWRITE NEEDED: False

QUESTION: если вторичка убитая совсем это сильно дороже?
SOURCE MODE: website_only
FIRST TOP SCORE: 0.24049956
REWRITE NEEDED: True
REWRITTEN: Стоимость ремонта в полностью изношенной вторичке
SECOND TOP SCORE: 0.45010513

QUESTION: Сколько стоит демонтаж плитки?
SOURCE MODE: all
FIRST TOP SCORE: 0.65144473
REWRITE NEEDED: False

QUESTION: Объясни квантовую механику
SOURCE MODE: all
FIRST TOP SCORE: 0.32732037
REWRITE NEEDED: True
REWRITTEN: Объяснение основ квантовой механики в ремонте квартир
SECOND TOP SCORE: 0.44038004

